# Financial Loan Data Cleaning
This notebook cleans the `financial_loan.xlsx` dataset and saves the cleaned output to the `Cleaned_Data` folder.

In [1]:
from pathlib import Path
import pandas as pd


base_path = Path.cwd().resolve().parent
raw_file = base_path / '01_Dataset' / 'Raw_Data' / 'financial_loan.xlsx'
output_file = base_path / '01_Dataset' / 'Cleaned_Data' / 'financial_loan_cleaned.csv'

date_columns = [
    'issue_date',
    'last_credit_pull_date',
    'last_payment_date',
    'next_payment_date',
]

numeric_columns = [
    'annual_income',
    'dti',
    'installment',
    'int_rate',
    'loan_amount',
    'total_acc',
    'total_payment',
]

text_columns = [
    'address_state',
    'application_type',
    'emp_length',
    'emp_title',
    'grade',
    'home_ownership',
    'loan_status',
    'purpose',
    'sub_grade',
    'term',
    'verification_status',
]

In [2]:
df = pd.read_excel(raw_file)
print('Raw shape:', df.shape)
df.head()

Raw shape: (38576, 24)


,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,sub_grade,term,verification_status,annual_income,dti,installment,int_rate,loan_amount,total_acc,total_payment
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,2021-02-11,2021-09-13,2021-04-13,...,C4,60 months,Source Verified,30000.0,0.0100,59.83,0.1527,2500,4,1009
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,2021-01-01,2021-12-14,2021-01-15,...,E1,36 months,Source Verified,48000.0,0.0535,109.43,0.1864,3000,4,3939
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,2021-01-05,2021-12-12,2021-01-09,...,C5,36 months,Not Verified,50000.0,0.2088,421.65,0.1596,12000,11,3522
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,B,MORTGAGE,2021-02-25,2021-12-12,2021-03-12,...,B2,60 months,Source Verified,42000.0,0.0540,97.06,0.1065,4500,9,4911
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,A,MORTGAGE,2021-01-01,2021-12-14,2021-01-15,...,A1,36 months,Verified,83000.0,0.0231,106.53,0.0603,3500,28,3835


In [6]:
def clean_financial_loan_data(dataframe: pd.DataFrame) -> pd.DataFrame:
    cleaned = dataframe.copy()
    cleaned.columns = [column.strip().lower() for column in cleaned.columns]
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)

    for column in date_columns:
        if column in cleaned.columns:
            cleaned[column] = pd.to_datetime(cleaned[column], errors='coerce')

    for column in numeric_columns:
        if column in cleaned.columns:
            cleaned[column] = pd.to_numeric(cleaned[column], errors='coerce')

    for column in text_columns:
        if column in cleaned.columns:
            cleaned[column] = cleaned[column].astype('string').str.strip()

    if 'address_state' in cleaned.columns:
        cleaned['address_state'] = cleaned['address_state'].str.upper()

    if 'emp_title' in cleaned.columns:
        cleaned['emp_title'] = cleaned['emp_title'].fillna('Unknown')

    if 'term' in cleaned.columns:
        cleaned['term_months'] = (
            cleaned['term'].astype('string').str.extract(r'(\d+)').astype('Int64')
        )

    if 'issue_date' in cleaned.columns:
        cleaned['issue_year'] = cleaned['issue_date'].dt.year.astype('Int64')
        cleaned['issue_month'] = cleaned['issue_date'].dt.month.astype('Int64')

    if {'loan_amount', 'total_payment'}.issubset(cleaned.columns):
        cleaned['payment_gap'] = cleaned['total_payment'] - cleaned['loan_amount']

    return cleaned

In [7]:
cleaned_df = clean_financial_loan_data(df)
print('Cleaned shape:', cleaned_df.shape)
print()
print('Missing values after cleaning:')
print(cleaned_df.isna().sum().sort_values(ascending=False))
cleaned_df.head()

Cleaned shape: (38576, 28)

Missing values after cleaning:
id                       0
address_state            0
application_type         0
emp_length               0
emp_title                0
grade                    0
home_ownership           0
issue_date               0
last_credit_pull_date    0
last_payment_date        0
loan_status              0
next_payment_date        0
member_id                0
purpose                  0
sub_grade                0
term                     0
verification_status      0
annual_income            0
dti                      0
installment              0
int_rate                 0
loan_amount              0
total_acc                0
total_payment            0
term_months              0
issue_year               0
issue_month              0
payment_gap              0
dtype: int64


,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,dti,installment,int_rate,loan_amount,total_acc,total_payment,term_months,issue_year,issue_month,payment_gap
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,2021-02-11,2021-09-13,2021-04-13,...,0.0100,59.83,0.1527,2500,4,1009,60,2021,2,-1491
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,2021-01-01,2021-12-14,2021-01-15,...,0.0535,109.43,0.1864,3000,4,3939,36,2021,1,939
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,2021-01-05,2021-12-12,2021-01-09,...,0.2088,421.65,0.1596,12000,11,3522,36,2021,1,-8478
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,B,MORTGAGE,2021-02-25,2021-12-12,2021-03-12,...,0.0540,97.06,0.1065,4500,9,4911,60,2021,2,411
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,A,MORTGAGE,2021-01-01,2021-12-14,2021-01-15,...,0.0231,106.53,0.0603,3500,28,3835,36,2021,1,335


In [8]:
output_file.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(output_file, index=False)
print(f'Cleaned dataset saved to: {output_file}')

Cleaned dataset saved to: D:\GitHub\Data-Analyst-Projects\01_Business_Analytics\Financial_Performance_Dashboard\01_Dataset\Cleaned_Data\financial_loan_cleaned.csv
